
## Table of Contents

- [Environment Configuration](#environment-configuration)
- [Park Accessibility Analysis](#park-accessibility-analysis)
- [Identifying Underserved Communities](#identifying-underserved-communities)
- [Population Analysis](#population-analysis)
- [Proximity Analysis](#proximity-analysis)
- [Priority Scoring and Ranking](#priority-scoring-and-ranking)
- [Final Priority Ranking](#final-priority-ranking)
- [Population Density Quality Control](#population-density-quality-control)
- [Results Validation](#results-validation)
- [Results and Discussion](#results-and-discussion)
- [Limitations and Future Improvements](#limitations-and-future-improvements)
- [Conclusion](#conclusion)



# Washington, DC | Park Accessibility & Priority Analysis

**Spatial analysis and automated priority assessment using ArcPy**

## Project Overview

Access to public parks is an important consideration in urban planning, particularly in neighborhoods where residents may have limited access to green space. This project evaluates the geographic distribution of usable parks across Washington, DC and identifies communities that may benefit from additional park investment.

The analysis combines park boundaries, Census block groups, and 2024 American Community Survey (ACS) population estimates to identify underserved areas and rank preliminary planning priorities.

Using ArcPy, the workflow automates spatial screening, proximity analysis, population density calculations, and weighted priority scoring. The results provide a consistent, repeatable approach for evaluating potential areas for future park investment.

## Analytical Approach

The analysis uses an 800-meter straight-line distance threshold to evaluate park accessibility. Census block groups that fall entirely outside the dissolved park buffers are classified as underserved.

Populated underserved areas are evaluated using two criteria:

- **Population density (60%)** to prioritize areas with greater concentrations of residents.
- **Distance to the nearest usable park (40%)** to account for relative geographic isolation.

Both criteria are normalized against their respective maximum values before calculating the weighted priority score.

The resulting rankings identify areas for further planning investigation rather than confirmed locations for park construction.

## Technical Environment

- **GIS platform:** ArcGIS Pro
- **Programming:** Python and ArcPy
- **Spatial reference:** NAD83 / Maryland (EPSG:26985)
- **Data storage:** Esri file geodatabase
- **Primary inputs:** Usable park polygons, Census block groups, and ACS population estimates
- **Outputs:** Underserved block groups, representative priority points, normalized scores, and ranked priority locations



## Environment Configuration

The analysis uses ArcPy with a local Esri file geodatabase. The workspace is configured to access the prepared park and Census block-group datasets, while a separate geodatabase stores outputs generated by this notebook.

All distance-based operations use NAD83 / Maryland (EPSG:26985), a projected coordinate system with meters as its linear unit. The analysis parameters are defined centrally to make the workflow easier to maintain and adapt to different distance thresholds or scoring weights.


In [65]:

from pathlib import Path
import arcpy

# Project workspace and source data
project_root = Path(
    r"C:\Users\armeh\OneDrive\Desktop\Nahid\ArcGIS GitHub"
    r"\DC_Park_Accessibility_Planner"
)

source_gdb = (
    project_root
    / "arcgis_pro"
    / "DC_Park_Accessibility_Planner"
    / "DC_Park_Accessibility_Planner.gdb"
)

output_gdb = project_root / "arcpy" / "DC_Park_ArcPy_Results.gdb"

# Analysis parameters
buffer_distance_m = 800
population_weight = 0.60
distance_weight = 0.40
target_wkid = 26985

# Input feature classes
parks = str(source_gdb / "DC_Parks_Usable")
block_groups = str(source_gdb / "DC_BlockGroups_Population")

# Configure ArcPy
arcpy.env.overwriteOutput = True
arcpy.env.outputCoordinateSystem = arcpy.SpatialReference(target_wkid)

# Validate inputs before processing
for name, dataset in {
    "Usable parks": parks,
    "Census block groups": block_groups
}.items():
    if not arcpy.Exists(dataset):
        raise FileNotFoundError(f"{name} not found: {dataset}")

    sr = arcpy.Describe(dataset).spatialReference
    if sr.factoryCode != target_wkid:
        raise ValueError(
            f"{name} uses WKID {sr.factoryCode}; expected {target_wkid}."
        )

    print(f"{name}: {arcpy.management.GetCount(dataset)[0]} features")

# Create a dedicated output geodatabase
if not arcpy.Exists(str(output_gdb)):
    arcpy.management.CreateFileGDB(
        str(output_gdb.parent),
        output_gdb.name
    )

arcpy.env.workspace = str(output_gdb)

print(f"\nOutput workspace: {output_gdb}")
print("Environment configured successfully.")

Usable parks: 147 features
Census block groups: 571 features

Output workspace: C:\Users\armeh\OneDrive\Desktop\Nahid\ArcGIS GitHub\DC_Park_Accessibility_Planner\arcpy\DC_Park_ArcPy_Results.gdb
Environment configured successfully.



## Park Accessibility Analysis

Park accessibility is evaluated using an 800-meter straight-line distance threshold around existing usable parks in Washington, DC.

The analysis creates buffers around park polygons and dissolves overlapping areas into a single accessibility coverage layer. Dissolving the buffers eliminates internal overlaps and provides a consistent geographic reference for identifying communities outside the selected distance threshold.

The resulting coverage represents proximity to park boundaries, rather than walking distance along streets or actual park entrance accessibility. This distinction is important when interpreting the results for urban planning.


In [66]:

# Create a dissolved accessibility buffer around usable parks.
park_accessibility = str(output_gdb / "DC_Park_Accessibility_800m")

arcpy.analysis.Buffer(
    in_features=parks,
    out_feature_class=park_accessibility,
    buffer_distance_or_field=f"{buffer_distance_m} Meters",
    line_side="FULL",
    line_end_type="ROUND",
    dissolve_option="ALL",
    method="PLANAR"
)

# Confirm the accessibility coverage was generated.
buffer_count = int(arcpy.management.GetCount(park_accessibility)[0])
buffer_sr = arcpy.Describe(park_accessibility).spatialReference

print(f"Input park polygons: {arcpy.management.GetCount(parks)[0]}")
print(f"Buffer distance: {buffer_distance_m:,} meters")
print(f"Dissolved accessibility features: {buffer_count}")
print(f"Output coordinate system: {buffer_sr.name}")
print(f"Output: {park_accessibility}")

if buffer_count != 1:
    raise RuntimeError(
        "Expected one dissolved buffer feature. "
        "Review the buffer output before continuing."
    )

Input park polygons: 147
Buffer distance: 800 meters
Dissolved accessibility features: 1
Output coordinate system: NAD_1983_StatePlane_Maryland_FIPS_1900
Output: C:\Users\armeh\OneDrive\Desktop\Nahid\ArcGIS GitHub\DC_Park_Accessibility_Planner\arcpy\DC_Park_ArcPy_Results.gdb\DC_Park_Accessibility_800m



## Identifying Underserved Communities

Census block groups are screened against the dissolved 800-meter park accessibility coverage to identify areas without geographic overlap.

The analysis uses an inverted spatial intersection to select block groups that do not intersect the accessibility buffer. Under this definition, an underserved block group must lie entirely outside the buffered area.

This approach provides a consistent geographic screening method, but it does not measure accessibility for individual households. Block groups that partially overlap the buffer are excluded from the underserved classification, even when some residents may live beyond the 800-meter threshold.

The selected block groups are preserved as a separate feature class for demographic analysis and subsequent priority assessment.


In [67]:

# Identify Census block groups with no intersection
# with the dissolved park accessibility coverage.

underserved_areas = str(
    output_gdb / "DC_Underserved_BlockGroups_800m"
)

block_group_layer = "block_groups_accessibility_lyr"

# Create a temporary feature layer for spatial selection.
arcpy.management.MakeFeatureLayer(
    in_features=block_groups,
    out_layer=block_group_layer
)

# Invert the intersection to retain only block groups
# entirely outside the 800-meter park buffer.
arcpy.management.SelectLayerByLocation(
    in_layer=block_group_layer,
    overlap_type="INTERSECT",
    select_features=park_accessibility,
    selection_type="NEW_SELECTION",
    invert_spatial_relationship="INVERT"
)

# Preserve the selected features in the ArcPy output geodatabase.
arcpy.management.CopyFeatures(
    in_features=block_group_layer,
    out_feature_class=underserved_areas
)

# Validate the geographic screening results.
total_block_groups = int(
    arcpy.management.GetCount(block_groups)[0]
)

underserved_count = int(
    arcpy.management.GetCount(underserved_areas)[0]
)

print(f"Census block groups evaluated: {total_block_groups:,}")
print(f"Underserved block groups: {underserved_count:,}")
print(f"Within/intersecting buffer: {total_block_groups - underserved_count:,}")
print(f"Output: {underserved_areas}")

if underserved_count != 6:
    print(
        "Validation warning: The result differs from the "
        "six underserved block groups identified in ModelBuilder."
    )

# Remove the temporary selection layer.
arcpy.management.Delete(block_group_layer)

Census block groups evaluated: 571
Underserved block groups: 6
Within/intersecting buffer: 565
Output: C:\Users\armeh\OneDrive\Desktop\Nahid\ArcGIS GitHub\DC_Park_Accessibility_Planner\arcpy\DC_Park_ArcPy_Results.gdb\DC_Underserved_BlockGroups_800m


<Result 'true'>


## Population Analysis

The underserved block groups are evaluated using population estimates from the 2024 American Community Survey (ACS) five-year dataset.

Population totals provide context for the number of residents represented by the selected geographic areas. Block groups with no recorded population are retained in the underserved-area dataset but excluded from priority scoring, which focuses on populated communities.

Population density is calculated using each block group's area in square kilometers. This allows the analysis to compare residential concentration across areas of different sizes.

Because the geographic screening identifies entire block groups outside the 800-meter buffers, the population totals describe residents of those selected block groups. They should not be interpreted as a household-level measure of walking accessibility.


In [68]:

# Inspect demographic fields in the underserved block groups.
population_fields = [
    field for field in arcpy.ListFields(underserved_areas)
    if "pop" in field.name.lower()
    or "area" in field.name.lower()
]

print("Population and area-related fields:\n")

for field in population_fields:
    print(f"{field.name:<35} {field.type}")


Population and area-related fields:

AREALAND                            BigInteger
AREAWATER                           BigInteger
Total_Population                    Integer
Shape_Area                          Double


In [69]:

# Summarize the population represented by underserved block groups.
population_field = "Total_Population"

with arcpy.da.SearchCursor(
    underserved_areas,
    [population_field]
) as cursor:
    populations = [
        int(row[0] or 0)
        for row in cursor
    ]

total_underserved_population = sum(populations)
populated_group_count = sum(pop > 0 for pop in populations)

print(f"Underserved block groups: {len(populations):,}")
print(f"Total underserved population: {total_underserved_population:,}")
print(f"Populated underserved groups: {populated_group_count:,}")

# Compare the results with the existing ModelBuilder analysis.
expected_population = 4237
expected_populated_groups = 4

assert total_underserved_population == expected_population, (
    "Population total differs from the ModelBuilder result."
)

assert populated_group_count == expected_populated_groups, (
    "Populated block-group count differs from the ModelBuilder result."
)

print("\nDemographic validation passed.")


Underserved block groups: 6
Total underserved population: 4,237
Populated underserved groups: 4

Demographic validation passed.


In [70]:

# Retain populated underserved areas for priority assessment.
priority_areas = str(output_gdb / "DC_Populated_Priority_Areas_800m")

population_layer = "populated_underserved_lyr"

arcpy.management.MakeFeatureLayer(
    underserved_areas,
    population_layer,
    where_clause=f"{population_field} > 0"
)

arcpy.management.CopyFeatures(
    population_layer,
    priority_areas
)

arcpy.management.Delete(population_layer)

# Store polygon area and population density as explicit attributes.
existing_fields = {field.name for field in arcpy.ListFields(priority_areas)}

if "Area_km2" not in existing_fields:
    arcpy.management.AddField(priority_areas, "Area_km2", "DOUBLE")

if "Pop_Density" not in existing_fields:
    arcpy.management.AddField(priority_areas, "Pop_Density", "DOUBLE")

with arcpy.da.UpdateCursor(
    priority_areas,
    ["SHAPE@", population_field, "Area_km2", "Pop_Density"]
) as cursor:
    for geometry, population, _, _ in cursor:
        area_km2 = geometry.getArea("PLANAR", "SQUAREKILOMETERS")

        density = (
            float(population) / area_km2
            if area_km2 > 0 else None
        )

        cursor.updateRow((
            geometry,
            population,
            area_km2,
            density
        ))

print(f"Populated priority areas: {arcpy.management.GetCount(priority_areas)[0]}")
print(f"Output: {priority_areas}")


Populated priority areas: 4
Output: C:\Users\armeh\OneDrive\Desktop\Nahid\ArcGIS GitHub\DC_Park_Accessibility_Planner\arcpy\DC_Park_ArcPy_Results.gdb\DC_Populated_Priority_Areas_800m



## Proximity Analysis

Representative locations are generated for the populated underserved block groups to support a consistent comparison of their proximity to existing parks.

A point is placed inside each polygon using ArcGIS Pro's Feature To Point tool with the inside option enabled. Unlike a geometric centroid, which may fall outside an irregular polygon, this method ensures that each representative location lies within its associated block group.

The Near tool then measures the shortest straight-line distance from each representative point to the boundary of the nearest usable park polygon.

These distances are used as one component of the priority scoring model. They do not represent pedestrian travel distance, actual park entrances, or suitable parcels for future park development.


In [71]:

# Generate representative points within populated priority polygons.
priority_centers = str(
    output_gdb / "DC_Park_Priority_Centers_ArcPy"
)

arcpy.management.FeatureToPoint(
    in_features=priority_areas,
    out_feature_class=priority_centers,
    point_location="INSIDE"
)

# Measure the nearest straight-line distance to a usable park.
arcpy.analysis.Near(
    in_features=priority_centers,
    near_features=parks,
    method="PLANAR"
)

# Verify the representative points and proximity measurements.
center_count = int(arcpy.management.GetCount(priority_centers)[0])

if center_count != 4:
    raise RuntimeError(
        f"Expected four priority centers; found {center_count}."
    )

distances = []

with arcpy.da.SearchCursor(
    priority_centers,
    ["OID@", "NEAR_DIST"]
) as cursor:
    for object_id, distance in cursor:
        if distance is None or distance < 0:
            raise RuntimeError(
                f"No valid nearest park found for feature {object_id}."
            )

        distances.append(distance)

print(f"Representative priority locations: {center_count}")
print("\nDistance to nearest usable park:")

for index, distance in enumerate(sorted(distances), start=1):
    print(f"  Location {index}: {distance:,.1f} meters")

print(f"\nOutput: {priority_centers}")


Representative priority locations: 4

Distance to nearest usable park:
  Location 1: 967.1 meters
  Location 2: 1,106.0 meters
  Location 3: 1,260.7 meters
  Location 4: 1,396.7 meters

Output: C:\Users\armeh\OneDrive\Desktop\Nahid\ArcGIS GitHub\DC_Park_Accessibility_Planner\arcpy\DC_Park_ArcPy_Results.gdb\DC_Park_Priority_Centers_ArcPy



## Priority Scoring and Ranking

A weighted multi-criteria scoring model is used to evaluate and rank underserved communities based on population density and distance to the nearest usable park.

Population density receives a 60% weight to emphasize areas with higher residential concentration, while park distance receives a 40% weight to account for geographic isolation. This weighting balances potential population benefit with the need to improve geographic accessibility.

Both criteria are normalized to a 0–100 scale relative to their maximum observed values among the candidate areas.

The final priority score is calculated as:

**Priority Score = (0.60 × Normalized Population Density) + (0.40 × Normalized Park Distance)**

Higher scores indicate greater preliminary planning priority. The weighting reflects a planning assumption rather than an empirically established optimum, and the resulting scores are relative to the current candidate areas.

In [72]:

# Identify the maximum observed values for score normalization.
with arcpy.da.SearchCursor(
    priority_centers,
    ["Pop_Density", "NEAR_DIST"]
) as cursor:
    criteria = [
        (density, distance)
        for density, distance in cursor
    ]

if not criteria or any(
    density is None or density < 0 or distance is None or distance < 0
    for density, distance in criteria
):
    raise ValueError("Priority criteria contain missing or invalid values.")

max_density = max(density for density, _ in criteria)
max_distance = max(distance for _, distance in criteria)

if max_density <= 0 or max_distance <= 0:
    raise ValueError("Normalization requires positive maximum values.")

# Create scoring fields if they do not already exist.
score_fields = {
    "Density_Score": "DOUBLE",
    "Distance_Score": "DOUBLE",
    "Priority_Score": "DOUBLE"
}

existing_fields = {
    field.name for field in arcpy.ListFields(priority_centers)
}

for field_name, field_type in score_fields.items():
    if field_name not in existing_fields:
        arcpy.management.AddField(
            priority_centers,
            field_name,
            field_type
        )

# Calculate normalized criteria and the weighted priority score.
with arcpy.da.UpdateCursor(
    priority_centers,
    [
        "Pop_Density",
        "NEAR_DIST",
        "Density_Score",
        "Distance_Score",
        "Priority_Score"
    ]
) as cursor:
    for density, distance, _, _, _ in cursor:
        density_score = (density / max_density) * 100
        distance_score = (distance / max_distance) * 100

        priority_score = (
            population_weight * density_score
            + distance_weight * distance_score
        )

        cursor.updateRow((
            density,
            distance,
            density_score,
            distance_score,
            priority_score
        ))

print(f"Maximum population density: {max_density:,.2f} residents/km²")
print(f"Maximum park distance: {max_distance:,.1f} meters")
print("\nPriority scores calculated successfully.")


Maximum population density: 11,890.66 residents/km²
Maximum park distance: 1,396.7 meters

Priority scores calculated successfully.



## Final Priority Ranking

The populated underserved areas are ranked according to their weighted priority scores, with the highest-scoring area assigned Rank 1.

The ranking combines residential concentration and geographic distance from existing usable parks. Population density carries greater weight, reflecting the project's emphasis on identifying underserved areas where additional green space could potentially benefit more residents within a smaller geographic area.

The final output preserves the demographic attributes, proximity measurements, normalized criteria, and priority scores for each representative location.

The results provide a transparent basis for comparing preliminary planning priorities. They are not a substitute for evaluating land availability, pedestrian connectivity, development feasibility, or community needs.


In [73]:

import pandas as pd

# Preserve the scored locations as a separate final output.
final_ranked = str(output_gdb / "DC_Park_Final_Ranked_ArcPy")

arcpy.management.CopyFeatures(
    priority_centers,
    final_ranked
)

# Add a field for the final priority ranking.
existing_fields = {
    field.name for field in arcpy.ListFields(final_ranked)
}

if "Priority_Rank" not in existing_fields:
    arcpy.management.AddField(
        final_ranked,
        "Priority_Rank",
        "SHORT"
    )

# Sort by descending score, with ObjectID as a stable tie-breaker.
with arcpy.da.SearchCursor(
    final_ranked,
    ["OID@", "Priority_Score"]
) as cursor:
    ranked_features = sorted(
        cursor,
        key=lambda row: (-row[1], row[0])
    )

rank_lookup = {
    object_id: rank
    for rank, (object_id, _) in enumerate(ranked_features, start=1)
}

with arcpy.da.UpdateCursor(
    final_ranked,
    ["OID@", "Priority_Rank"]
) as cursor:
    for object_id, _ in cursor:
        cursor.updateRow((
            object_id,
            rank_lookup[object_id]
        ))

# Prepare a concise table for reviewing the final rankings.
result_fields = [
    "Priority_Rank",
    "Total_Population",
    "Pop_Density",
    "NEAR_DIST",
    "Density_Score",
    "Distance_Score",
    "Priority_Score"
]

with arcpy.da.SearchCursor(final_ranked, result_fields) as cursor:
    results = pd.DataFrame(
        list(cursor),
        columns=result_fields
    )

results = results.sort_values("Priority_Rank")

display(
    results.round({
        "Pop_Density": 2,
        "NEAR_DIST": 1,
        
        "Density_Score": 2,
        "Distance_Score": 2,
        "Priority_Score": 2
    })
)

print(f"Ranked priority locations: {len(results)}")
print(f"Final feature class: {final_ranked}")


,Priority_Rank,Total_Population,Pop_Density,NEAR_DIST,Density_Score,Distance_Score,Priority_Score
2,1,738,11890.66,967.1,100.00,69.24,87.70
1,2,696,6627.65,1106.0,55.74,79.19,65.12
0,3,693,4464.23,1396.7,37.54,100.00,62.53
3,4,2110,4254.88,1260.7,35.78,90.26,57.58


Ranked priority locations: 4
Final feature class: C:\Users\armeh\OneDrive\Desktop\Nahid\ArcGIS GitHub\DC_Park_Accessibility_Planner\arcpy\DC_Park_ArcPy_Results.gdb\DC_Park_Final_Ranked_ArcPy



## Population Density Quality Control

Population density calculations are checked against Census-reported land area to assess the consistency of the geometric measurements.

The comparison helps verify that the projected polygon areas used in the ArcPy workflow produce reasonable density estimates. Minor differences may result from differences between geometric polygon area and Census land-area measurements.


In [74]:

# Compare geometric area with Census-reported land area.
with arcpy.da.SearchCursor(
    priority_areas,
    [
        "Total_Population",
        "Area_km2",
        "AREALAND",
        "Pop_Density"
    ]
) as cursor:
    records = []

    for population, geometry_area, census_land_area, density in cursor:
        census_area_km2 = (
            census_land_area / 1_000_000
            if census_land_area is not None else None
        )

        records.append({
            "Population": population,
            "Geometry_Area_km2": geometry_area,
            "Census_Land_Area_km2": census_area_km2,
            "ArcPy_Density": density,
            "Census_Land_Density": (
                population / census_area_km2
                if census_area_km2 and census_area_km2 > 0
                else None
            )
        })

area_comparison = pd.DataFrame(records)

display(area_comparison.round(4))


,Population,Geometry_Area_km2,Census_Land_Area_km2,ArcPy_Density,Census_Land_Density
0,693,0.1552,0.1553,4464.2254,4463.7106
1,696,0.1050,0.1050,6627.6455,6626.9936
2,738,0.0621,0.0621,11890.6640,11889.4187
3,2110,0.4959,0.4954,4254.8770,4259.3135



## Results Validation

The ArcPy results are compared with the independently developed ArcGIS ModelBuilder workflow to verify consistency between the two implementations.

Both workflows use the same source datasets, an 800-meter park accessibility threshold, and a weighted priority model consisting of 60% population density and 40% distance to the nearest usable park.

The comparison matches candidate locations using Census block-group GEOIDs and checks population density, park distance, normalized scoring components, final priority scores, and rankings.

Agreement between the two workflows provides an additional quality-control check on the spatial analysis and scoring calculations.


In [75]:

# Validate final ArcPy results against the ModelBuilder output.
import pandas as pd

modelbuilder_results = str(
    source_gdb / "DC_Park_Final_Ranked_800m"
)

if not arcpy.Exists(modelbuilder_results):
    raise FileNotFoundError(
        "ModelBuilder results are required for cross-validation."
    )

fields_mb = [
    "GEOID", "Pop_Density", "NEAR_DIST",
    "Population_Score", "Distance_Score",
    "Priority_Score", "Final_Rank"
]

fields_arcpy = [
    "GEOID", "Pop_Density", "NEAR_DIST",
    "Density_Score", "Distance_Score",
    "Priority_Score", "Priority_Rank"
]

with arcpy.da.SearchCursor(modelbuilder_results, fields_mb) as cursor:
    mb = pd.DataFrame(list(cursor), columns=fields_mb)

with arcpy.da.SearchCursor(final_ranked, fields_arcpy) as cursor:
    ap = pd.DataFrame(list(cursor), columns=fields_arcpy)

validation = mb.merge(
    ap,
    on="GEOID",
    suffixes=("_MB", "_ArcPy"),
    validate="one_to_one"
)

assert len(validation) == 4, "Candidate count mismatch."

checks = {
    "Population density": ("Pop_Density_MB", "Pop_Density_ArcPy"),
    "Park distance": ("NEAR_DIST_MB", "NEAR_DIST_ArcPy"),
    "Population score": ("Population_Score", "Density_Score"),
    "Distance score": ("Distance_Score_MB", "Distance_Score_ArcPy"),
    "Priority score": ("Priority_Score_MB", "Priority_Score_ArcPy"),
    "Priority rank": ("Final_Rank", "Priority_Rank")
}

for label, (mb_field, ap_field) in checks.items():
    difference = (
        validation[mb_field] - validation[ap_field]
    ).abs()

    tolerance = 0.01 if label != "Priority rank" else 0

    if not (difference <= tolerance).all():
        raise AssertionError(f"{label} does not match ModelBuilder.")

print("Validation passed: ArcPy and ModelBuilder results match.")
print(f"Validated priority locations: {len(validation)}")

display(
    validation[
        ["GEOID", "Priority_Rank", "Priority_Score_ArcPy"]
    ].sort_values("Priority_Rank").round(2)
)


Validation passed: ArcPy and ModelBuilder results match.
Validated priority locations: 4


,GEOID,Priority_Rank,Priority_Score_ArcPy
0,110010082002,1,87.70
1,110010022023,2,65.12
2,110010095102,3,62.53
3,110010022021,4,57.58



## Results and Discussion

The analysis evaluated 147 usable parks and 571 Census block groups across Washington, DC. Using an 800-meter straight-line accessibility threshold, six block groups were identified as entirely outside the buffered park coverage.

These six block groups represent an estimated 4,237 residents based on the 2024 ACS five-year population data. Four block groups contained residents and were included in the priority assessment.

The highest-ranked location received a priority score of approximately 87.70, reflecting the greatest population density among the four candidates. The remaining locations received scores of approximately 65.12, 62.53, and 57.58.

Although one candidate contained 2,110 residents, it ranked fourth because the model emphasizes population density rather than total population. This distinction illustrates how the selection of evaluation criteria influences planning priorities.

A comparison with an alternative 70% population density and 30% park distance weighting produced the same ranking of the four candidate locations. This provides a limited sensitivity check, although broader testing would be needed to evaluate the robustness of the model under other weighting assumptions.

The results identify geographic areas that warrant further planning investigation rather than specific parcels suitable for park construction.



## Limitations and Future Improvements

This analysis provides a preliminary assessment of park accessibility and investment priorities. Several limitations should be considered when interpreting the results.

**Euclidean distance:** Accessibility is measured using straight-line distances to park boundaries. Actual walking distances may differ because of street connectivity, physical barriers, and the location of park entrances.

**Block-group screening:** Only Census block groups entirely outside the 800-meter accessibility buffers are classified as underserved. This approach may overlook residents in partially covered block groups who also experience limited park access.

**Population estimates:** The analysis uses ACS five-year population estimates at the block-group level. These estimates do not describe the exact distribution of residents within each polygon.

**Weighting assumptions:** The 60% population density and 40% park distance weights represent a planning judgment. Alternative weighting strategies could produce different scores or rankings.

**Site feasibility:** Representative points are analytical locations, not proposed construction sites. Land ownership, zoning, environmental conditions, acquisition costs, and community preferences have not been evaluated.

Future improvements could incorporate pedestrian network analysis, park entrances, finer-scale population distribution, socioeconomic indicators, park capacity and quality, and land suitability constraints. A broader sensitivity analysis could also evaluate how changes in criteria and weights affect the final rankings.

## Conclusion

The ArcPy workflow provides a repeatable approach for identifying and ranking geographically underserved communities in Washington, DC.

By combining park proximity, Census population data, spatial analysis, and weighted scoring, the project demonstrates how GIS can support evidence-based urban planning decisions.

The results offer a starting point for further investigation into park investment opportunities while maintaining a clear distinction between analytical priorities and feasible development sites.
